In [ ]:

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU — go to Runtime > Change Runtime Type > T4 GPU")


from google.colab import drive
drive.mount('/content/drive')
! pip install ultralytics 


import cv2
import os
import time
import math
import numpy as np
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display

VIDEO_PATH        = "/content/drive/MyDrive/IPD/test3.mov"  
OUTPUT_PATH       = "/content/drive/MyDrive/IPD/posture1.mp4"
SKIP_FRAMES       = 2     
CONFIDENCE        = 0.20
NMS_THRESH        = 0.35
MIN_PERSON_HEIGHT = 80     
PREVIEW_EVERY     = 15     
GRID_ROWS            = 3
GRID_COLS            = 3
ZONE_ALERT_THRESHOLD = 2

NOSE           = 0
LEFT_SHOULDER  = 5;  RIGHT_SHOULDER = 6
LEFT_ELBOW     = 7;  RIGHT_ELBOW    = 8
LEFT_WRIST     = 9;  RIGHT_WRIST    = 10
LEFT_HIP       = 11; RIGHT_HIP      = 12
LEFT_KNEE      = 13; RIGHT_KNEE     = 14
LEFT_ANKLE     = 15; RIGHT_ANKLE    = 16

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

print("[INFO] Loading models...")
detect_model = YOLO("yolo11x.pt")
pose_model   = YOLO("yolov8m-pose.pt")
print("[INFO] Models loaded.")


COLOR_WALKING  = (0, 0, 255)
COLOR_STANDING = (0, 255, 0)
COLOR_UNKNOWN  = (128, 128, 128)
COLOR_ALERT    = (0, 0, 255)
SKELETON_COLOR = (0, 200, 100)


def get_kp(kp_array, idx, min_conf=0.4):
    x, y, c = kp_array[idx]
    return (float(x), float(y)) if c >= min_conf else None


def angle_between(a, b, c):
    ax, ay = a[0]-b[0], a[1]-b[1]
    cx, cy = c[0]-b[0], c[1]-b[1]
    dot = ax*cx + ay*cy
    mag = math.sqrt(ax**2+ay**2) * math.sqrt(cx**2+cy**2)
    if mag == 0:
        return 0
    return math.degrees(math.acos(max(-1, min(1, dot/mag))))


def enhance_frame(frame):
    lab     = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l       = clahe.apply(l)
    frame   = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    frame   = cv2.fastNlMeansDenoisingColored(frame, None, 5, 5, 7, 21)
    kernel  = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    return cv2.filter2D(frame, -1, kernel)

def classify_posture(kp):
    l_hip   = get_kp(kp, LEFT_HIP)
    r_hip   = get_kp(kp, RIGHT_HIP)
    l_knee  = get_kp(kp, LEFT_KNEE)
    r_knee  = get_kp(kp, RIGHT_KNEE)
    l_ankle = get_kp(kp, LEFT_ANKLE)
    r_ankle = get_kp(kp, RIGHT_ANKLE)

    if not all([l_hip, r_hip, l_knee, r_knee]):
        return 'unknown'

    hip_width = abs(r_hip[0] - l_hip[0]) if r_hip and l_hip else 50
    score     = 0

    l_ka = angle_between(l_hip, l_knee, l_ankle) if l_ankle else None
    r_ka = angle_between(r_hip, r_knee, r_ankle) if r_ankle else None
    if (l_ka and 100 < l_ka < 165) or (r_ka and 100 < r_ka < 165):
        score += 1

    hip_cx  = (l_hip[0]  + r_hip[0])  / 2
    knee_cx = (l_knee[0] + r_knee[0]) / 2
    if abs(knee_cx - hip_cx) > hip_width * 0.3:
        score += 1

    if l_ankle and r_ankle:
        if abs(l_ankle[0] - r_ankle[0]) > hip_width * 0.5:
            score += 1

    if abs(l_knee[1] - r_knee[1]) > 15:
        score += 1

    if score >= 2:
        return 'walking'
    elif score == 1:
        return 'unknown'
    return 'standing'


def draw_skeleton(frame, kp, color):
    for s, e in SKELETON:
        p1 = get_kp(kp, s)
        p2 = get_kp(kp, e)
        if p1 and p2:
            cv2.line(frame, (int(p1[0]),int(p1[1])),
                     (int(p2[0]),int(p2[1])), color, 2)
    for i in range(len(kp)):
        p = get_kp(kp, i)
        if p:
            cv2.circle(frame, (int(p[0]),int(p[1])), 3, (0,255,0), -1)


def get_zone(cx, cy, w, h):
    col = min(int(cx/w*GRID_COLS), GRID_COLS-1)
    row = min(int(cy/h*GRID_ROWS), GRID_ROWS-1)
    return row, col


def draw_zone_grid(frame, zone_walking, zone_total):
    h, w   = frame.shape[:2]
    zone_w = w // GRID_COLS
    zone_h = h // GRID_ROWS

    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            walking = zone_walking[row][col]
            total   = zone_total[row][col]
            x1 = col*zone_w; y1 = row*zone_h
            x2 = x1+zone_w;  y2 = y1+zone_h

            if walking == 0:
                color, alpha = (255,255,255), 0.03
            elif walking < ZONE_ALERT_THRESHOLD:
                color, alpha = (0,255,255), 0.12
            else:
                color, alpha = (0,0,255), 0.25

            overlay = frame.copy()
            cv2.rectangle(overlay, (x1,y1), (x2,y2), color, -1)
            cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 1)

            cv2.putText(frame, f"Z{row*GRID_COLS+col+1}",
                        (x1+6,y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            if total > 0:
                cv2.putText(frame, f"{total}p",
                            (x1+6,y1+40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            if walking > 0:
                cv2.putText(frame, f"{walking}w",
                            (x1+6,y1+60), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            COLOR_WALKING, 1)
            if walking >= ZONE_ALERT_THRESHOLD:
                cv2.putText(frame, "ALERT",
                            (x1+6,y1+80), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                            (0,0,255), 2)


def process_frame(frame):
    h, w = frame.shape[:2]

    enhanced = enhance_frame(frame)

    det     = detect_model(enhanced, conf=CONFIDENCE, classes=[0], verbose=False)
    boxes   = det[0].boxes

    zone_walking = [[0]*GRID_COLS for _ in range(GRID_ROWS)]
    zone_total   = [[0]*GRID_COLS for _ in range(GRID_ROWS)]
    walking_count  = 0
    standing_count = 0

    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf            = float(box.conf[0])
        person_h        = y2 - y1
        cx              = (x1+x2)//2
        cy              = (y1+y2)//2
        row, col        = get_zone(cx, cy, w, h)
        zone_total[row][col] += 1

        if person_h < MIN_PERSON_HEIGHT:
            cv2.rectangle(enhanced, (x1,y1), (x2,y2), COLOR_UNKNOWN, 1)
            continue

        pad  = 15
        cx1  = max(0, x1-pad); cy1 = max(0, y1-pad)
        cx2  = min(w, x2+pad); cy2 = min(h, y2+pad)
        crop = enhanced[cy1:cy2, cx1:cx2]

        if crop.size == 0:
            continue

        pose_res = pose_model(crop, conf=0.5, verbose=False)

        if (pose_res[0].keypoints is None or
                len(pose_res[0].keypoints) == 0):
            cv2.rectangle(enhanced, (x1,y1), (x2,y2), COLOR_UNKNOWN, 1)
            continue

        kp = pose_res[0].keypoints.data[0].cpu().numpy()
        kp_full      = kp.copy()
        kp_full[:,0] = kp[:,0] + cx1
        kp_full[:,1] = kp[:,1] + cy1

        posture = classify_posture(kp_full)

        if posture == 'walking':
            color = COLOR_WALKING
            walking_count += 1
            zone_walking[row][col] += 1
        elif posture == 'standing':
            color = COLOR_STANDING
            standing_count += 1
        else:
            color = COLOR_UNKNOWN

        cv2.rectangle(enhanced, (x1,y1), (x2,y2), color, 2)
        cv2.putText(enhanced, f"{posture.upper()} {conf:.0%}",
                    (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
        draw_skeleton(enhanced, kp_full, color)

    draw_zone_grid(enhanced, zone_walking, zone_total)

    alert_zones = sum(1 for r in zone_walking for c in r
                      if c >= ZONE_ALERT_THRESHOLD)

    if walking_count > 0:
        cv2.rectangle(enhanced, (0,0), (w,45), (0,0,180), -1)
        cv2.putText(enhanced, f"  ⚠  {walking_count} WALKING DETECTED  ⚠",
                    (10,32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

    cv2.putText(enhanced, f"Walking: {walking_count}",
                (10, h-60), cv2.FONT_HERSHEY_SIMPLEX, 0.65, COLOR_WALKING, 2)
    cv2.putText(enhanced, f"Standing: {standing_count}",
                (10, h-35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, COLOR_STANDING, 2)
    cv2.putText(enhanced, f"Alert Zones: {alert_zones}",
                (w-230, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (0,0,255) if alert_zones > 0 else (200,200,200), 2)

    return enhanced, walking_count, standing_count, alert_zones


def run():
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print("[ERROR] Cannot open video. Check your path.")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    orig_w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"[INFO] Video: {orig_w}x{orig_h} @ {fps:.1f}fps | {total_frames} frames")
    print(f"[INFO] Output: {OUTPUT_PATH}\n")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (orig_w, orig_h))

    progress       = widgets.IntProgress(
                        value=0, min=0, max=total_frames,
                        description='Processing:',
                        style={'description_width': 'initial'},
                        layout=widgets.Layout(width='600px'))
    label          = widgets.Label(value="Starting...")
    preview_widget = widgets.Image(format='jpeg', width=750)
    display(widgets.VBox([progress, label, preview_widget]))

    frame_count = 0
    last_frame  = None
    start_time  = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        if frame_count % SKIP_FRAMES != 0:
            if last_frame is not None:
                out.write(last_frame)
            continue

        annotated, walking, standing, alerts = process_frame(frame)

        elapsed     = time.time() - start_time
        current_fps = frame_count / elapsed if elapsed > 0 else 0
        eta_sec     = int((total_frames-frame_count)/current_fps) if current_fps > 0 else 0

        cv2.putText(annotated,
                    f"Frame: {frame_count}/{total_frames} | FPS: {current_fps:.1f}",
                    (10, orig_h-10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (150,150,150), 1)

        out.write(annotated)
        last_frame = annotated.copy()

        progress.value = frame_count
        label.value    = (f"Frame {frame_count}/{total_frames} | "
                          f"{current_fps:.1f} FPS | "
                          f"ETA: {eta_sec//60}m{eta_sec%60}s | "
                          f"Walking: {walking} | Standing: {standing} | "
                          f"Alerts: {alerts}")

        if frame_count % PREVIEW_EVERY == 0:
            _, buf = cv2.imencode('.jpg', annotated,
                                  [cv2.IMWRITE_JPEG_QUALITY, 75])
            preview_widget.value = buf.tobytes()

    cap.release()
    out.release()

    total_time = time.time() - start_time
    print(f"\n[DONE] {frame_count} frames in {total_time:.1f}s "
          f"({frame_count/total_time:.1f} FPS avg)")
    print(f"[DONE] Saved to: {OUTPUT_PATH}")

run()

Sat Mar 14 03:23:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----